In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Inisialisasi Library

In [6]:
# ==========================================
# CELL 1: INSTALL & SETUP
# ==========================================
import torch
import subprocess
import sys
import os

# 1. Cek GPU
if torch.cuda.is_available():
    print(f"✅ GPU Terdeteksi: {torch.cuda.get_device_name(0)}")
    gpu_available = True
else:
    print("❌ GPU TIDAK TERDETEKSI!")
    gpu_available = False

# 2. Install Library Utama (satu per satu agar error lebih mudah dilacak)
print("\nMenginstall library...")
!pip install -q sentence-transformers
!pip install -q python-terrier
!pip install -q pyarabic
!pip install -q tashaphyne
!pip install -q pandas

# 3. Install FAISS
try:
    if gpu_available:
        print("Mencoba install faiss-gpu...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faiss-gpu"])
        print("✅ faiss-gpu berhasil diinstall.")
    else:
        raise Exception("No GPU")
except:
    print("⚠️ Menginstall faiss-cpu...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"])

# 4. Clone / Pull Repository
if os.path.exists('skripsi-clir-code'):
    print("Repository ditemukan. Melakukan pull...")
    !cd skripsi-clir-code && git pull
else:
    print("Cloning repository...")
    !git clone https://github.com/syifaurrr/skripsi-clir-code.git

# 5. Setup Path & Import
REPO_PATH = './skripsi-clir-code'
SRC_PATH = os.path.join(REPO_PATH, 'src')
sys.path.append(SRC_PATH)

import pyterrier as pt
if not pt.started():
    pt.init()
print("✅ PyTerrier Started!")

✅ GPU Terdeteksi: Tesla T4

Menginstall library...
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.8/208.8 kB 4.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 24.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.5/251.5 kB 6.6 MB/s eta 0:00:0000:01
Mencoba install faiss-gpu...


ERROR: Could not find a version that satisfies the requirement faiss-gpu (from versions: none)
ERROR: No matching distribution found for faiss-gpu


⚠️ Menginstall faiss-cpu...
Repository ditemukan. Melakukan pull...
Already up to date.


/tmp/ipykernel_55/787247093.py:51: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():


terrier-assemblies 5.11 jar-with-dependencies not found, downloading to /root/.pyterrier...


...

Done
terrier-python-helper 0.0.8 jar not found, downloading to /root/.pyterrier...


https://repo1.maven.org/maven2/org/terrier/terrier-python-helper/0.0.8/terrier-python-helper-0.0.8.jar: 100%|██████████| 36.6k/36.6k [00:00<00:00, 26.6MB/s]

Done


✅ PyTerrier Started!


Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/tmp/ipykernel_55/787247093.py:52: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


# Inisialisasi Faiss Retriever

In [7]:
# ==========================================
# CELL 2: CLASS FAISS RETRIEVER
# ==========================================
import faiss
import numpy as np
import pandas as pd
import pyterrier as pt
from sentence_transformers import SentenceTransformer

class FaissRetriever(pt.Transformer):
    def __init__(self, model_name, index_path=None, doc_df=None, batch_size=32):
        self.model = SentenceTransformer(model_name)
        
        # Patch untuk mencegah error parameter tokenizer
        if hasattr(self.model, 'tokenizer') and 'token_type_ids' in self.model.tokenizer.model_input_names:
            self.model.tokenizer.model_input_names.remove('token_type_ids')
            
        self.index = None
        self.doc_map = {} 
        self.batch_size = batch_size
        
        if doc_df is not None:
            self._index_docs(doc_df)
            
    def _index_docs(self, doc_df):
        print(f"Encoding {len(doc_df)} dokumen ke vektor (bisa agak lama)...")
        # PENTING: Untuk Skenario 3, pastikan kolom yang masuk adalah teks Arab yang sudah di-preprocess
        docs = doc_df['text_processed'].tolist()
        docnos = doc_df['docno'].tolist()
        
        embeddings = self.model.encode(docs, batch_size=self.batch_size, show_progress_bar=True, convert_to_numpy=True)
        faiss.normalize_L2(embeddings)
        
        dim = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dim)
        self.index.add(embeddings)
        
        self.doc_map = {i: docno for i, docno in enumerate(docnos)}
        print("✅ Indexing selesai.")

    def transform(self, queries):
        q_texts = queries['query'].tolist()
        q_embeddings = self.model.encode(q_texts, batch_size=self.batch_size, show_progress_bar=False, convert_to_numpy=True)
        faiss.normalize_L2(q_embeddings)
        
        k = 1000
        scores, ids = self.index.search(q_embeddings, k)
        
        results = []
        for i, qid in enumerate(queries['qid']):
            for j in range(k):
                idx = ids[i][j]
                if idx != -1:
                    results.append({
                        'qid': str(qid),
                        'docno': self.doc_map[idx],
                        'score': scores[i][j],
                        'rank': j + 1
                    })
        return pd.DataFrame(results)

# Loading Data 

In [8]:
# ==========================================
# CELL 3: PERSIAPAN DATA (DOKUMEN & KUERI ARAB)
# ==========================================
# Tanpa preprocessing apapun — teks Arab mentah langsung dipakai

DATA_PATH = './skripsi-clir-code/data'
RAW_DOCS = os.path.join(DATA_PATH, 'raw/fathul_muin.csv')
QUERY_ARAB = os.path.join(DATA_PATH, 'queries/Kueri_Assesor_Zona2 - queries_arab.csv')
QRELS_PATH = os.path.join(DATA_PATH, 'queries/Kueri_Assesor_Zona2 - qrels.csv')

# [!] Sesuaikan nama kolom ini dengan yang ada di CSV
KOLOM_NMT    = 'query_nmt'   # kolom kueri Google Translate
KOLOM_LLM    = 'query_llm'   # kolom kueri Gemini AI

print("1. Memuat & Preprocessing Dokumen...")
df_docs = pd.read_csv(RAW_DOCS)
df_docs['text_processed'] = df_docs['text']  # tanpa normalisasi, tanpa stemming
df_docs['docno'] = df_docs['docno'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)

print("2. Memuat Kueri Bahasa Arab (Google NMT & Gemini LLM)...")
df_queries_arab = pd.read_csv(QUERY_ARAB)

# Cek kolom yang tersedia
print(f"   Kolom tersedia: {df_queries_arab.columns.tolist()}")

# Buat topics NMT
topics_nmt = df_queries_arab[['qid', KOLOM_NMT]].rename(columns={KOLOM_NMT: 'query'}).copy()
topics_nmt['qid'] = topics_nmt['qid'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)

# Buat topics LLM
topics_llm = df_queries_arab[['qid', KOLOM_LLM]].rename(columns={KOLOM_LLM: 'query'}).copy()
topics_llm['qid'] = topics_llm['qid'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)

print(f"   Topics NMT: {len(topics_nmt)} kueri")
print(f"   Topics LLM: {len(topics_llm)} kueri")

print("3. Memuat Qrels...")
qrels = pd.read_csv(QRELS_PATH)
if 'doc_no' in qrels.columns:
    qrels.rename(columns={'doc_no': 'docno'}, inplace=True)
elif 'doc_id' in qrels.columns:
    qrels.rename(columns={'doc_id': 'docno'}, inplace=True)

qrels['qid']   = qrels['qid'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
qrels['docno'] = qrels['docno'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
qrels['label'] = qrels['label'].astype(int)

print("4. Memuat Metadata Tipe Kueri...")
QUERY_INDO = os.path.join(DATA_PATH, 'queries/Kueri_Assesor_Zona2 - queries_indo.csv')
df_queries_indo = pd.read_csv(QUERY_INDO)
df_queries_indo['qid'] = df_queries_indo['qid'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)

print("✅ Persiapan Data Selesai!")

1. Memuat & Preprocessing Dokumen...
2. Memuat Kueri Bahasa Arab (Google NMT & Gemini LLM)...
   Kolom tersedia: ['qid', 'query', 'query_nmt', 'query_llm']
   Topics NMT: 51 kueri
   Topics LLM: 51 kueri
3. Memuat Qrels...
4. Memuat Metadata Tipe Kueri...
✅ Persiapan Data Selesai!


# Evaluasi

In [9]:
# ==========================================
# CELL 4: EVALUASI ARADPR (NMT vs LLM)
# ==========================================
from IPython.display import display
import numpy as np

print("Membangun Index Dense menggunakan model Monolingual: abdoelsayed/AraDPR...")
ARADPR_MODEL_NAME = 'abdoelsayed/AraDPR'

retriever_aradpr = FaissRetriever(
    model_name=ARADPR_MODEL_NAME,
    doc_df=df_docs[['docno', 'text_processed']],
    batch_size=16
)

# Daftar metrik evaluasi lengkap
eval_metrics_list = ["recip_rank", "success_10", "success_20", "success_50", "success_100"]

print("\n1. Menjalankan Evaluasi Per-Kueri (perquery=True)...")

# Eksperimen 1: AraDPR + Google NMT
hasil_nmt_perq = pt.Experiment(
    [retriever_aradpr],
    topics_nmt,
    qrels,
    eval_metrics=eval_metrics_list,
    names=["AraDPR (Google NMT)"],
    validate="ignore",
    perquery=True
)

# Eksperimen 2: AraDPR + Gemini LLM
hasil_llm_perq = pt.Experiment(
    [retriever_aradpr],
    topics_llm,
    qrels,
    eval_metrics=eval_metrics_list,
    names=["AraDPR (Gemini LLM)"],
    validate="ignore",
    perquery=True
)

# Gabungkan hasil NMT dan LLM
hasil_eval_perq = pd.concat([hasil_nmt_perq, hasil_llm_perq], ignore_index=True)

# Pivot: metrik jadi kolom
hasil_pivot = hasil_eval_perq.pivot_table(
    index=['name', 'qid'],
    columns='measure',
    values='value'
).reset_index()
hasil_pivot.columns.name = None

# Pastikan semua kolom metrik ada
for m in eval_metrics_list:
    if m not in hasil_pivot.columns:
        hasil_pivot[m] = 0.0

print("\n2. Memuat Metadata Tipe Kueri dan Menggabungkannya...")
hasil_pivot['qid'] = hasil_pivot['qid'].astype(str)
df_queries_indo['qid'] = df_queries_indo['qid'].astype(str)

hasil_dengan_tipe = pd.merge(
    hasil_pivot,
    df_queries_indo[['qid', 'query_type', 'query']],
    on='qid',
    how='left'
)

print("\n✅ Evaluasi selesai.")

Membangun Index Dense menggunakan model Monolingual: abdoelsayed/AraDPR...


No sentence-transformers model found with name abdoelsayed/AraDPR. Creating a new one with mean pooling.


config.json:   0%|          | 0.00/658 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/711M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Encoding 639 dokumen ke vektor (bisa agak lama)...


Batches:   0%|          | 0/40 [00:00<?, ?it/s]

✅ Indexing selesai.

1. Menjalankan Evaluasi Per-Kueri (perquery=True)...

2. Memuat Metadata Tipe Kueri dan Menggabungkannya...

✅ Evaluasi selesai.


In [10]:
# ==========================================
# CELL 5: HASIL EVALUASI LENGKAP
# ==========================================

# ---- 1. HASIL KESELURUHAN (OVERALL) ----
print("=" * 50)
print("🎯 HASIL KESELURUHAN (NMT vs LLM)")
print("=" * 50)

overall_metrics = hasil_dengan_tipe.groupby('name')[eval_metrics_list].mean().reset_index()

for k in [10, 20, 50, 100]:
    overall_metrics[f'success_{k} (%)'] = (overall_metrics[f'success_{k}'] * 100).round(2)
overall_metrics['recip_rank'] = overall_metrics['recip_rank'].round(4)

display(overall_metrics[['name', 'recip_rank', 'success_10 (%)', 'success_20 (%)', 'success_50 (%)', 'success_100 (%)']])

# ---- 2. HASIL BERDASARKAN TIPE KUERI ----
print("\n" + "=" * 50)
print("📊 HASIL BERDASARKAN TIPE KUERI")
print("=" * 50)

per_type_metrics = hasil_dengan_tipe.groupby(['name', 'query_type'])[eval_metrics_list].mean().reset_index()

for k in [10, 20, 50, 100]:
    per_type_metrics[f'success_{k} (%)'] = (per_type_metrics[f'success_{k}'] * 100).round(2)
per_type_metrics['recip_rank'] = per_type_metrics['recip_rank'].round(4)

display(per_type_metrics[['name', 'query_type', 'recip_rank', 'success_10 (%)', 'success_20 (%)', 'success_50 (%)', 'success_100 (%)']])

# ---- 3. ANALISIS DETAIL PER-KUERI (HIT/MISS) ----
print("\n" + "=" * 50)
print("🔍 ANALISIS DETAIL PER-KUERI")
print("=" * 50)

detail_kueri = hasil_dengan_tipe.copy()

detail_kueri['rank_ditemukan'] = detail_kueri['recip_rank'].apply(
    lambda x: int(round(1 / x)) if x > 0 else "Tidak Ketemu"
)

for k in [10, 20, 50, 100]:
    detail_kueri[f'Hit@{k}'] = detail_kueri[f'success_{k}'].apply(
        lambda x: '✅ Hit' if x == 1.0 else '❌ Miss'
    )

detail_kueri['qid_int'] = pd.to_numeric(detail_kueri['qid'], errors='coerce').fillna(0).astype(int)
detail_kueri = detail_kueri.sort_values(by=['name', 'query_type', 'qid_int'])

kolom_tampil = ['name', 'qid', 'query_type', 'query', 'rank_ditemukan', 'Hit@10', 'Hit@20', 'Hit@50', 'Hit@100']
tabel_detail_final = detail_kueri[kolom_tampil]

display(tabel_detail_final.head(15))

# ---- 4. SIMPAN CSV ----
output_dir = os.path.join(DATA_PATH, 'results')
os.makedirs(output_dir, exist_ok=True)

tabel_detail_final.to_csv(os.path.join(output_dir, 'skenario3_detail_per_kueri_nmt_vs_llm.csv'), index=False)
per_type_metrics.to_csv(os.path.join(output_dir, 'skenario3_tipe_kueri_nmt_vs_llm.csv'), index=False)
overall_metrics.to_csv(os.path.join(output_dir, 'skenario3_overall_nmt_vs_llm.csv'), index=False)

print("\n📑 Seluruh laporan berhasil disimpan dalam format CSV!")

🎯 HASIL KESELURUHAN (NMT vs LLM)


,name,recip_rank,success_10 (%),success_20 (%),success_50 (%),success_100 (%)
0,AraDPR (Gemini LLM),0.1387,23.53,37.25,45.10,58.82
1,AraDPR (Google NMT),0.0429,11.76,19.61,35.29,50.98



📊 HASIL BERDASARKAN TIPE KUERI


,name,query_type,recip_rank,success_10 (%),success_20 (%),success_50 (%),success_100 (%)
0,AraDPR (Gemini LLM),1,0.2269,35.29,47.06,47.06,52.94
1,AraDPR (Gemini LLM),2,0.1017,23.53,47.06,52.94,76.47
2,AraDPR (Gemini LLM),3,0.0875,11.76,17.65,35.29,47.06
3,AraDPR (Google NMT),1,0.0394,5.88,23.53,29.41,41.18
4,AraDPR (Google NMT),2,0.0541,23.53,23.53,41.18,64.71
5,AraDPR (Google NMT),3,0.0352,5.88,11.76,35.29,47.06



🔍 ANALISIS DETAIL PER-KUERI


,name,qid,query_type,query,rank_ditemukan,Hit@10,Hit@20,Hit@50,Hit@100
0,AraDPR (Gemini LLM),1,1,Memakai gamis saat ihram,15,❌ Miss,✅ Hit,✅ Hit,✅ Hit
33,AraDPR (Gemini LLM),4,1,Syarat-syarat penerima zakat,11,❌ Miss,✅ Hit,✅ Hit,✅ Hit
48,AraDPR (Gemini LLM),7,1,Barang belum diterima sudah dijual kembali,3,✅ Hit,✅ Hit,✅ Hit,✅ Hit
1,AraDPR (Gemini LLM),10,1,Kewajiban orang yang ghasab,118,❌ Miss,❌ Miss,❌ Miss,❌ Miss
4,AraDPR (Gemini LLM),13,1,Penetapan awal bulan ramadhan,1,✅ Hit,✅ Hit,✅ Hit,✅ Hit
7,AraDPR (Gemini LLM),16,1,Syarat wajib puasa,7,✅ Hit,✅ Hit,✅ Hit,✅ Hit
10,AraDPR (Gemini LLM),19,1,syarat barang yang sah disewakan,121,❌ Miss,❌ Miss,❌ Miss,❌ Miss
14,AraDPR (Gemini LLM),22,1,mabuk perjalanan saat berpuasa,321,❌ Miss,❌ Miss,❌ Miss,❌ Miss
17,AraDPR (Gemini LLM),25,1,barang yang tidak boleh dipinjamkan,1,✅ Hit,✅ Hit,✅ Hit,✅ Hit
20,AraDPR (Gemini LLM),28,1,jual beli dengan ijab qabul main-main,156,❌ Miss,❌ Miss,❌ Miss,❌ Miss



📑 Seluruh laporan berhasil disimpan dalam format CSV!
